# Lab 1 · Your First Network: the PyTorch training loop
**MSACL · DS301 Deep Learning · Segment 1 · Lab 1**

Welcome to your first hands-on lab! This is a ~50-minute session, and the goal is simple:
get comfortable with **Colab** and **PyTorch**, and run one small neural network through the exact
**train → evaluate** loop you saw in Lecture 2.

> **You fill in just two short cells — everything else runs for you.**
> Look for the two cells marked **YOUR TURN ✏️**; run every other cell exactly as it is.

We'll train a network to predict a person's biological **sex** from their **serum metabolite panel**
(189 measured metabolites) — the kind of tabular readout your instrument already produces.

## Running this notebook in Colab
- A **notebook** is a list of *cells*. A cell holds either text (like this) or code.
- To run a code cell, click it and press **Shift + Enter** (or the ▶ button). Run the cells **in order, top to bottom**.
- The first run downloads the data and sets up PyTorch — that takes a few seconds.
- If anything gets tangled, use **Runtime → Restart and run all** to start fresh.

No GPU is needed for this lab; the default CPU runtime is plenty.

In [ ]:
# Read and run — no need to edit.
# --- import the tools we'll use ---
import numpy as np                # numpy: fast math on big arrays of numbers
import pandas as pd               # pandas: reads spreadsheets/tables (our .xlsx data)
import torch                      # PyTorch: builds and trains the neural network
import torch.nn as nn            # nn: ready-made network pieces (layers, losses)
import matplotlib.pyplot as plt  # matplotlib: draws the loss curve

# Fix the random seeds so everyone gets the same numbers on every run.
np.random.seed(0)                 # seed numpy (used when we shuffle for the split)
torch.manual_seed(0)              # seed PyTorch (used when weights are first set)
print("PyTorch version:", torch.__version__)  # confirm PyTorch loaded

## About the data: MTBLS90

Before we train anything, let's understand what we're feeding the network.

**MTBLS90** is a public **serum LC–MS metabolomics** dataset — essentially the kind of readout
your own instrument produces: for each blood-serum sample, a mass spectrometer measures the
abundance of many small molecules (*metabolites*). We use it because it is real clinical-lab data,
it is freely downloadable with no sign-up, and it is big enough to train on yet small enough to
run in this lab.

- **Source:** MetaboLights study **MTBLS90**; the Excel file is mirrored by the CIMCB teaching
  repository (Mendez, Broadhurst, Reinke *et al.*). Public, open data.
- **Size:** **968 samples** (patients) × **189 metabolite features**.
- **Label:** biological **sex** — a balanced binary target (**485 male / 483 female**).
- **Missing values:** **none** (0 NaNs) — no cleanup needed.

### How the table is laid out
When we load the `Data` sheet we get a table (a pandas *DataFrame*) with these columns:

| Column group | What it is | Use in this lab |
|---|---|---|
| `Idx`, `SampleID` | row number and sample name (bookkeeping) | ignored |
| `Class` | 1 = male, 0 = female | **the label** (what we predict) |
| `Sex` | the same info as text (`"M"` / `"F"`) | ignored (we use `Class`) |
| `M1`, `M2`, …, `M189` | the 189 metabolite measurements | **the features** (network input) |

So: **features = the 189 `M#` columns**, **label = `Class`**. The `M#` labels are codes; the
human-readable metabolite names live in a separate `Peak` sheet in the same file (we don't need
them to train — that's just where you'd look them up).

## Step 1 · Load the data *(runs for you)*
We download the Excel file straight from the web and read its `Data` sheet into a table.

In [ ]:
# Read and run — no need to edit.
# The dataset lives online as an Excel (.xlsx) file; pandas can read it directly from a URL.
url = "https://raw.githubusercontent.com/CIMCB/MetabComparisonBinaryML/master/notebooks/data/MTBLS90.xlsx"
df = pd.read_excel(url, sheet_name="Data")   # read the 'Data' sheet into a DataFrame
print("Table shape:", df.shape)              # (rows, columns) = (968 samples, 193 columns)
df.head(3)                                    # peek at the first 3 rows

In [ ]:
# Read and run — no need to edit.
# Build the list of the 189 feature column names: "M1", "M2", ..., "M189".
feature_cols = [f"M{i}" for i in range(1, 190)]   # range(1, 190) gives the numbers 1..189

# X = the inputs: pull out just the metabolite columns and turn them into a numeric array.
X = df[feature_cols].to_numpy("float32")          # shape (968, 189)

# y = the label: 1 = male, 0 = female.
y = df["Class"].to_numpy("float32")               # shape (968,)

print("X (inputs):", X.shape)                     # (968, 189)
print("y (labels):", y.shape,
      "| counts [female, male]:", np.bincount(y.astype(int)))  # ~[483, 485], nearly balanced

Here's what a single **sample** looks like — 189 metabolite measurements for one person. A
"sample" is just a row of numbers, and the network sees exactly this.

In [ ]:
# Read and run — no need to edit.
# Let's look at ONE patient so you can see that a "sample" is just a row of 189 numbers.
sample_row = 0                                    # pick the first patient (row 0); try other rows if you like
sample_values = X[sample_row]                     # that patient's 189 RAW metabolite values (before normalizing)
sample_label = "male" if y[sample_row] == 1 else "female"  # translate the 0/1 Class into a word

plt.figure(figsize=(9, 3))                        # a wide, short figure fits 189 bars nicely
plt.bar(range(1, 190), sample_values)             # x = metabolite number 1..189, y = its measured value
plt.xlabel("metabolite number (M1 … M189)")       # x-axis: which metabolite
plt.ylabel("measured value (raw)")                # y-axis: its abundance, before normalization
plt.title(f"One serum sample (row {sample_row}, Class = {sample_label})")  # label with this patient's sex
plt.tight_layout()                                # keep the labels from being clipped
plt.show()                                         # display the figure

## Step 2 · Split and normalize — by hand *(runs for you)*
Two standard preparation steps. We do them **by hand with numpy** (no libraries) so you can see
exactly what happens:

1. **Split** the 968 samples into a **training set** (~80%, to learn from) and a **test set**
   (~20%, to check honestly on samples the model never saw). We shuffle the row order first, using
   a fixed seed so the split is identical on every run.
2. **Normalize (z-score)** each metabolite to roughly mean 0, spread 1: subtract the column's mean
   and divide by its standard deviation. Networks train far better when all inputs share a scale.

> **Why compute the mean/std on the training set only?** The test set stands in for *future,
> unseen patients*. If we used it to compute the scaling, we'd be leaking test information into
> training and our accuracy would be dishonestly high. So we learn mean/std from **train only**,
> then apply those same numbers to both train and test.

In [ ]:
# Read and run — no need to edit.
# --- 1. Shuffle the row order, then slice an 80/20 split ---
n_samples = X.shape[0]                    # 968 rows in total
indices = np.arange(n_samples)            # [0, 1, 2, ..., 967]
np.random.shuffle(indices)                # shuffle in place (seed fixed above -> same order every run)

n_train = int(0.8 * n_samples)            # keep 80% for training -> 774 rows
train_idx = indices[:n_train]             # first 774 shuffled rows -> training
test_idx  = indices[n_train:]             # the remaining 194 rows  -> testing

X_train, y_train = X[train_idx], y[train_idx]   # grab the training rows
X_test,  y_test  = X[test_idx],  y[test_idx]    # grab the test rows

# --- 2. Z-score normalize using TRAIN statistics only ---
mean = X_train.mean(axis=0)               # per-metabolite mean, computed on TRAIN only
std  = X_train.std(axis=0) + 1e-8         # per-metabolite std (+ tiny number avoids divide-by-zero)

X_train = (X_train - mean) / std          # apply the train mean/std to the training data
X_test  = (X_test  - mean) / std          # apply the SAME train mean/std to the test data (never refit!)

# --- 3. Convert numpy arrays into PyTorch tensors (what the network computes on) ---
X_train = torch.tensor(X_train, dtype=torch.float32)   # (774, 189)
X_test  = torch.tensor(X_test,  dtype=torch.float32)   # (194, 189)
# Labels become float COLUMN vectors of shape (N, 1) to match the network's (N, 1) output.
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)   # (774, 1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).reshape(-1, 1)   # (194, 1)

print("training samples:", X_train.shape[0], "| test samples:", X_test.shape[0])
print("y_train shape:", tuple(y_train.shape), "(a column vector, to match the model output)")

## Step 3 · Build the network — by hand *(runs for you)*
We define the network as a small **class**. Think of the class as a blueprint with two parts:

- `__init__` **lists the layers** (created once): `fc1` maps 189 inputs → 16 hidden neurons;
  `fc2` maps those 16 → 1 output.
- `forward` **describes the path** an input takes, step by step: `fc1` → **ReLU** (the "bend" from
  Lecture 1) → `fc2` → **sigmoid** (squash to a 0–1 probability).

It's the same tiny network as the slides (189 → 16 → 1), just written out explicitly so you can see
every layer. (`fc` stands for *fully connected*, another name for `nn.Linear`.)

We also need two more pieces: the **loss** — MSE, how wrong the guess is (Lecture 2; *Lecture 4
introduces the proper classification loss, cross-entropy*) — and the **optimizer** — Adam, the rule
that nudges the weights downhill.

In [ ]:
# Read and run — no need to edit.
class MetaboliteNet(nn.Module):          # our network, built on PyTorch's base class nn.Module
    def __init__(self):
        super().__init__()               # required: switch on the nn.Module machinery
        self.fc1 = nn.Linear(189, 16)    # layer 1: 189 metabolite inputs -> 16 hidden neurons
        self.relu = nn.ReLU()            # activation: the "bend" that lets the net learn curves
        self.fc2 = nn.Linear(16, 1)      # layer 2: 16 hidden neurons -> 1 output number
        self.sigmoid = nn.Sigmoid()      # squash that output into a probability between 0 and 1

    def forward(self, x):                # forward: how an input x flows through the network
        x = self.fc1(x)                  # 1. multiply by layer 1's weights (189 -> 16)
        x = self.relu(x)                 # 2. apply ReLU to the 16 hidden values
        x = self.fc2(x)                  # 3. multiply by layer 2's weights (16 -> 1)
        x = self.sigmoid(x)              # 4. squash to a 0..1 probability
        return x                         # hand back the prediction

model = MetaboliteNet()                  # create one network from the blueprint
print(model)                             # print its layers

In [ ]:
# Read and run — no need to edit.
loss_fn   = nn.MSELoss()                                    # loss: how wrong the guess is (Lecture 2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # optimizer: the rule that nudges weights downhill

## Step 4 · Train the network — YOUR TURN ✏️
This is the first cell you write. Fill in the **five lines of the training loop** from Lecture 2,
**in order**:

1. **forward** — make a guess with the model
2. **loss** — measure how wrong the guess is
3. **zero** — clear last step's gradients
4. **backward** — backprop the blame
5. **step** — nudge every weight

**Your Turn — the names you'll use** (these are the pieces, not the answers):
`model`, `X_train`, `y_train`, `loss_fn`, `optimizer`, and the methods
`.zero_grad()`, `.backward()`, `.step()`. Store the guess in `pred` and the loss in `loss`.

Stuck? The paper **hint sheet** lists multiple-choice options for each line. Run the cell when
you're done — the `assert` checks underneath will tell you if it worked.

In [ ]:
EPOCHS = 150
loss_history = []

for epoch in range(EPOCHS):
    # ========== YOUR CODE HERE ==========
    ...  # replace with your code
    loss_history.append(loss.item())   # (written for you) record the loss

# --- self-checks: do not edit ---
assert len(loss_history) == EPOCHS, "The loop should record one loss per epoch."
assert model.fc1.weight.grad is not None, "No gradients — did you call loss.backward()?"
assert loss_history[-1] < 0.5 * loss_history[0], \
    "The loss barely fell — did you call optimizer.step() to update the weights?"
print(f"first-epoch loss {loss_history[0]:.3f}  ->  last-epoch loss {loss_history[-1]:.3f}")
print("Nice — the loss fell. Your network is learning!")

## Step 5 · Watch it learn *(runs for you)*
Plot the loss at every epoch. You should see the **Lecture 2 loss curve**: a steep drop that
flattens out — proof the network is improving with each step.

In [ ]:
# Read and run — no need to edit.
plt.figure(figsize=(6, 4))                      # start a new figure
plt.plot(loss_history)                           # y = loss at each epoch (x = epoch number)
plt.xlabel("epoch"); plt.ylabel("loss (MSE)")    # label the axes
plt.title("Training loss — it should fall and flatten")
plt.show()                                        # display the figure

## Step 6 · Evaluate on the test set — YOUR TURN ✏️
Training accuracy can lie — the model has *seen* that data. The honest test is on the **held-out
test samples**. Fill in the evaluation:

- wrap the prediction in `with torch.no_grad():` (we're only checking, not learning — no gradients needed),
- turn the 0–1 probabilities into 0/1 predictions with a **0.5 threshold**,
- compute the fraction that match the true label.

**Your Turn — the names you'll use** (not the answers): `torch.no_grad()`, `model(X_test)`, the
threshold `0.5`, `y_test`, and `.float().mean()` to get the fraction correct. Store the result in
`test_acc`.

In [ ]:
# ========== YOUR CODE HERE ==========
...  # replace with your code

# --- self-checks: do not edit ---
assert 0.0 <= test_acc <= 1.0, "Accuracy must be a fraction between 0 and 1."
assert test_acc > 0.65, "Expected clearly above the 0.50 coin-flip baseline."
print(f"Test accuracy: {test_acc:.3f}    (coin-flip baseline: 0.50)")

## What just happened
You built and trained a real neural network — by hand, end to end — and evaluated it honestly:

- the **loss fell** as the network learned (Steps 4–5),
- it reached roughly **0.72–0.77 accuracy** on samples it had never seen — well above the **0.50**
  coin-flip baseline, so it found real signal in the metabolite panel.

Notice it isn't 100%: predicting sex from metabolites is genuinely hard, and the model also
*overfits* the training data a little (training loss keeps sinking while test accuracy plateaus).
Closing that train/test gap is exactly what **Lecture 4** is about.

## Optional stretch *(only if you're ahead)*
Curious what moves the result? Change **one** small thing and then **Runtime → Run all**:

- change `EPOCHS` in Step 4 (try `50` or `300`), **or**
- change the hidden width `16` in Step 3's model — edit **both** `nn.Linear(189, 16)` and
  `nn.Linear(16, 1)` to the same new number (try `4` or `64`).

Watch the loss curve and the test accuracy move. There's no single right answer — this is the
tuning you'll do for real in later labs.